In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Besoin Client 2 – Prédiction de l'âge des arbres

L'objectif est de développer un modèle capable de prédire l'âge (`age_estim`) d'un arbre à partir de ses caractéristiques.

> Comme l'âge est une valeur **continue**, il s'agit d'un problème de **régression** (et non de classification).


## Import des bibliothèques nécessaires


In [24]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Chargement des données


In [25]:
data = pd.read_csv(
    '/content/drive/MyDrive/Projet_BigDataIAetWeb/DataForIA2.csv',
    sep=";"
)

print("Shape :", data.shape)
data.head()


Shape : (9915, 13)


,clc_quartier,clc_secteur,haut_tot,haut_tronc,tronc_diam,fk_stadedev,fk_port,fk_pied,fk_situation,age_estim,fk_prec_estim,feuillage,remarquable
0,Quartier du Centre-Ville,Quai Gayant,6.0,2.0,37.0,Jeune,semi libre,gazon,Alignement,15,5.0,Feuillu,Non
1,Quartier Saint-Jean,Rue Demoustier,0.0,0.0,0.0,NaN,NaN,fosse arbre,Alignement,15,10.0,NaN,Non
2,Quartier du Vermandois,Stade Cepy,13.0,1.0,160.0,Adulte,semi libre,gazon,Groupe,50,10.0,Conifère,Non
3,Quartier du Centre-Ville,Rue Villebois Mareuil,12.0,3.0,116.0,Adulte,semi libre,gazon,Alignement,30,10.0,Feuillu,Non
4,Quartier de l'Europe,Square des Marronniers,16.0,3.0,150.0,Adulte,semi libre,gazon,Groupe,50,2.0,Feuillu,Non


## Préparation des données

Suppression des lignes contenant au moins un "NaN"

In [26]:
data = data.dropna()
print("Shape après suppression des NaN :", data.shape)

Shape après suppression des NaN : (9435, 13)


### Encodage des variables qualitatives

On encode uniquement les colonnes listées dans qualitative pour ne pas toucher aux colonnes déjà numériques.


In [27]:
le = LabelEncoder()

qualitative = ['feuillage', 'remarquable', 'fk_stadedev',
               'fk_pied', 'fk_port', 'clc_quartier', 'clc_secteur']

for col in qualitative:
    data[col] = le.fit_transform(data[col].astype(str))

data[qualitative].head()


,feuillage,remarquable,fk_stadedev,fk_pied,fk_port,clc_quartier,clc_secteur
0,1,0,1,6,10,7,133
2,0,0,0,6,10,8,266
3,1,0,0,6,10,7,198
4,1,0,0,6,10,6,263
5,1,0,0,6,8,6,6


### Séparation features / cible

On définit :
- **`X`** : les features (variables d'entrée du modèle)
- **`y`** : la cible à prédire (`age_estim`)

Puis on divise les données en **80% pour l'entraînement** et **20% pour le test**.  


In [28]:
FEATURES = ['tronc_diam', 'haut_tot', 'haut_tronc', 'fk_stadedev',
            'feuillage', 'remarquable', 'fk_pied', 'fk_port',
            'clc_quartier', 'clc_secteur']
TARGET = 'age_estim'

X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {X_train.shape} | Test : {X_test.shape}")


Train : (7548, 10) | Test : (1887, 10)


### Normalisation (StandardScaler)


In [29]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Sauvegarde du scaler – indispensable pour le script final
joblib.dump(scaler, "scaler_age.pkl")
print("Scaler sauvegardé.")


Scaler sauvegardé.


## Comparaison de plusieurs algorithmes de régression

On teste 4 algorithmes pour choisir le plus performant sur nos données.  
Les métriques utilisées sont :
- **MAE** *(Mean Absolute Error)* : erreur moyenne en années
- **RMSE** *(Root Mean Squared Error)* : pénalise davantage les grosses erreurs
- **R²** : proportion de la variabilité expliquée par le modèle


In [30]:
models = {
    "LinearRegression"      : LinearRegression(),
    "DecisionTreeRegressor" : DecisionTreeRegressor(random_state=42),
    "RandomForestRegressor" : RandomForestRegressor(random_state=42),
    "GradientBoosting"      : GradientBoostingRegressor(random_state=42),
}

print(f"{'Modèle':<30} | {'MAE':>6} | {'RMSE':>6} | {'R²':>7}")
print("-" * 60)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    print(f"{name:<30} | {mae:>6.2f} | {rmse:>6.2f} | {r2:>7.4f}")


Modèle                         |    MAE |   RMSE |      R²
------------------------------------------------------------
LinearRegression               |   8.05 |  11.31 |  0.6791
DecisionTreeRegressor          |   3.00 |   9.02 |  0.7959
RandomForestRegressor          |   2.68 |   6.05 |  0.9083
GradientBoosting               |   5.36 |   7.94 |  0.8420


## Analyse du R² en fonction du nombre de features

On évalue l'apport de chaque feature en ajoutant une variable à la fois.  
Cela permet de visualiser quelles colonnes **contribuent réellement** à la prédiction et à partir de combien de features le modèle se stabilise.

On utilise `RandomForestRegressor` car c'est l'algorithme le plus performant d'après la comparaison précédente.


In [31]:
print(f"{'Nbr Features':>12} |  R²")
print("-" * 82)

for i in range(1, len(FEATURES) + 1):
    variables = FEATURES[:i]

    X_loop = data[variables]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_loop, y, test_size=0.2, random_state=42
    )

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc  = sc.transform(X_te)

    rf = RandomForestRegressor(random_state=42)
    rf.fit(X_tr_sc, y_tr)

    r2 = r2_score(y_te, rf.predict(X_te_sc))
    print(f"{i:>12} | {r2:.4f}")


Nbr Features |  R²
----------------------------------------------------------------------------------
           1 | 0.5250
           2 | 0.5609
           3 | 0.6646
           4 | 0.7181
           5 | 0.7545
           6 | 0.7691
           7 | 0.7973
           8 | 0.8113
           9 | 0.8767
          10 | 0.9083


## Optimisation des hyperparamètres (GridSearchCV)

Le `GridSearchCV` teste **toutes les combinaisons possibles** des hyperparamètres définis dans `param_grid`.  
Pour chaque combinaison, une **validation croisée à 5 folds** est réalisée afin d'estimer les performances de façon robuste.

Les hyperparamètres explorés :
- `n_estimators` : nombre d'arbres dans la forêt
- `max_depth` : profondeur maximale de chaque arbre
- `min_samples_leaf` : nombre minimum d'échantillons dans une feuille



In [32]:
param_grid = {
    "n_estimators"  : [20, 100, 200],
    "max_depth"     : [None, 10, 20],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print("Meilleurs hyperparamètres :", grid_search.best_params_)
best_model = grid_search.best_estimator_


Fitting 5 folds for each of 27 candidates, totalling 135 fits
Meilleurs hyperparamètres : {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}


## Métriques finales du meilleur modèle

On évalue le modèle optimisé sur le jeu de **test** (données jamais vues pendant l'entraînement).


In [33]:
y_pred_best = best_model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred_best)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2   = r2_score(y_test, y_pred_best)

print("── Métriques du meilleur modèle ──")
print(f"  MAE  : {mae:.2f} ans")
print(f"  RMSE : {rmse:.2f} ans")
print(f"  R²   : {r2:.4f}")


── Métriques du meilleur modèle ──
  MAE  : 2.67 ans  → erreur moyenne de prédiction
  RMSE : 6.05 ans  → sensible aux grosses erreurs
  R²   : 0.9084    → proportion de variabilité expliquée


## Sauvegarde du modèle final


In [34]:
joblib.dump(best_model, "model_age_arbre.pkl")
print("Modèle sauvegardé : model_age_arbre.pkl")


Modèle sauvegardé : model_age_arbre.pkl
